In [3]:
import GEOparse
import pandas as pd
import os

# --- 1. CONFIGURATION & GLOBALS ---
gse_ids = ["GSE10325", "GSE38351", "GSE39088", "GSE49454", "GSE32591", "GSE45291", "GSE72747"]
genes_list = [
    "MTHFR", "TYMS", "MTR", "MTRR",
    "CYP1A1", "GSTM1", "GSTT1", "GSTP1",
    "TLR2", "TLR4", "TLR7", "TLR9",
    "STAT4"]

PROBES = {}
PLATFORM_BLACKLIST = ['GPL10558']

# --- 2. METADATA & FILTERING LOGIC ---
def get_metadata_labels(gsm):
    """
    Robustly identifies disease state and data type using full metadata.
    This fixes the GSE32591 blank title issue.
    """
    title = gsm.metadata.get('title', [''])[0]
    characteristics = " ".join(gsm.metadata.get('characteristics_ch1', []))
    description = " ".join(gsm.metadata.get('description', []))
    full_meta = f"{title} {characteristics} {description}".lower()

    # Identify Disease State
    if any(word in full_meta for word in ['healthy', 'control', 'normal']):
        disease_state = "Healthy Control"
    elif any(word in full_meta for word in ['sle', 'lupus']):
        disease_state = "SLE"
    else:
        disease_state = "Other/Unknown"

    # Identify Data Type (Professor's Requirement)
    sc_keywords = ['single cell', 'single-cell', 'scrna', '10x genomics', 'drop-seq']
    if any(word in full_meta for word in sc_keywords):
        data_type = "Single-Cell"
    else:
        data_type = "Bulk Tissue"

    return disease_state, data_type, title

# --- 3. PLATFORM & PROBE LOGIC ---
def process_platform(platform_id, genes_list):
    if platform_id in PLATFORM_BLACKLIST:
        return None

    print(f"Loading platform {platform_id}...")
    gpl = GEOparse.get_GEO(platform_id, destdir="./")
    annot = gpl.table

    # Map symbols (handling different possible column names in GEO)
    symbol_col = 'Gene Symbol' if 'Gene Symbol' in annot.columns else 'Symbol'
    if symbol_col not in annot.columns:
        # Fallback: find any column that looks like it has symbols
        potential_cols = [c for c in annot.columns if 'symbol' in c.lower()]
        symbol_col = potential_cols[0] if potential_cols else None

    gene_to_probe = {gene: [] for gene in genes_list}
    if symbol_col:
        result = annot[annot[symbol_col].isin(genes_list)]
        for _, row in result.iterrows():
            gene_to_probe[row[symbol_col]].append(row["ID"])

    return gene_to_probe

def process_gsm(gsm, genes_list):
    platform_id = gsm.metadata["platform_id"][0]
    if platform_id in PLATFORM_BLACKLIST:
        return None

    if platform_id not in PROBES:
        PROBES[platform_id] = process_platform(platform_id, genes_list)

    gene_to_probe = PROBES[platform_id]
    table = gsm.table

    # ID_REF is the standard GEO expression ID column
    id_col = "ID_REF" if "ID_REF" in table.columns else table.columns[0]

    # Flatten all probes for these genes to filter the table once
    all_target_probes = [p for probes in gene_to_probe.values() for p in probes]
    gsm_filtered = table[table[id_col].isin(all_target_probes)]

    gene_expression = {}
    for gene in genes_list:
        probes = gene_to_probe[gene]
        # Get values for this gene's probes and average them
        values = gsm_filtered[gsm_filtered[id_col].isin(probes)]["VALUE"].astype(float).tolist()
        if values:
            gene_expression[gene] = sum(values) / len(values)
        else:
            gene_expression[gene] = -1.0 # Indicator for missing data

    return gene_expression

# --- 4. MAIN EXECUTION LOOP ---
def process_gses(gse_ids, genes_list, filename="database_robust.csv"):
    with open(filename, "w") as file:
        # Header with new metadata columns
        file.write("GSE_ID,GSM_ID,GPL,Disease_State,Data_Type,title," + ",".join(genes_list) + "\n")

        for gse_id in gse_ids:
            print(f"\n>>> Processing Series: {gse_id}")
            try:
                gse = GEOparse.get_GEO(gse_id, destdir="./")
                for gsm_id, gsm in gse.gsms.items():
                    # 1. Extract and Filter Metadata
                    disease_state, data_type, title = get_metadata_labels(gsm)

                    if data_type == "Single-Cell":
                        continue # Professor's rule: no single-cell data

                    # 2. Process Expression
                    gene_expression = process_gsm(gsm, genes_list)
                    if gene_expression is None:
                        continue

                    # 3. Write to CSV
                    clean_title = title.replace(",", ";")
                    gpl = gsm.metadata['platform_id'][0]
                    expr_values = [str(gene_expression[g]) for g in genes_list]

                    row = [gse_id, gsm_id, gpl, disease_state, data_type, clean_title] + expr_values
                    file.write(",".join(row) + "\n")
            except Exception as e:
                print(f"Error in {gse_id}: {e}")

    print(f"\nDONE! Dataset saved to {filename}")

# --- 5. START ---
process_gses(gse_ids, genes_list)

21-Apr-2026 14:27:40 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
21-Apr-2026 14:27:40 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
21-Apr-2026 14:27:40 INFO GEOparse - Parsing ./GSE10325_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE10325_family.soft.gz: 
21-Apr-2026 14:27:40 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
21-Apr-2026 14:27:40 DEBUG GEOparse - SERIES: GSE10325
DEBUG:GEOparse:SERIES: GSE10325
21-Apr-2026 14:27:40 DEBUG GEOparse - PLATFORM: GPL96
DEBUG:GEOparse:PLATFORM: GPL96



>>> Processing Series: GSE10325


21-Apr-2026 14:27:41 DEBUG GEOparse - SAMPLE: GSM260886
DEBUG:GEOparse:SAMPLE: GSM260886
21-Apr-2026 14:27:41 DEBUG GEOparse - SAMPLE: GSM260887
DEBUG:GEOparse:SAMPLE: GSM260887
21-Apr-2026 14:27:41 DEBUG GEOparse - SAMPLE: GSM260888
DEBUG:GEOparse:SAMPLE: GSM260888
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260889
DEBUG:GEOparse:SAMPLE: GSM260889
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260890
DEBUG:GEOparse:SAMPLE: GSM260890
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260891
DEBUG:GEOparse:SAMPLE: GSM260891
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260892
DEBUG:GEOparse:SAMPLE: GSM260892
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260893
DEBUG:GEOparse:SAMPLE: GSM260893
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260894
DEBUG:GEOparse:SAMPLE: GSM260894
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260895
DEBUG:GEOparse:SAMPLE: GSM260895
21-Apr-2026 14:27:42 DEBUG GEOparse - SAMPLE: GSM260896
DEBUG:GEOparse:SAMPLE: GSM260896
21-Apr-2026 14:27:42 

Loading platform GPL96...


21-Apr-2026 14:27:48 DEBUG downloader - Total size: 0
DEBUG:GEOparse:Total size: 0
21-Apr-2026 14:27:48 DEBUG downloader - md5: None
DEBUG:GEOparse:md5: None
44.4MB [00:01, 23.8MB/s]
21-Apr-2026 14:27:50 DEBUG downloader - Moving /tmp/tmpdmp4wq69 to /content/GPL96.txt
DEBUG:GEOparse:Moving /tmp/tmpdmp4wq69 to /content/GPL96.txt
21-Apr-2026 14:27:50 DEBUG downloader - Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL96&form=text&view=full
DEBUG:GEOparse:Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL96&form=text&view=full
21-Apr-2026 14:27:50 INFO GEOparse - Parsing ./GPL96.txt: 
INFO:GEOparse:Parsing ./GPL96.txt: 
21-Apr-2026 14:27:50 DEBUG GEOparse - PLATFORM: GPL96
DEBUG:GEOparse:PLATFORM: GPL96
21-Apr-2026 14:27:52 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
21-Apr-2026 14:27:52 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/

...

Loading platform GPL13158...


21-Apr-2026 14:32:02 DEBUG downloader - Total size: 0
DEBUG:GEOparse:Total size: 0
21-Apr-2026 14:32:02 DEBUG downloader - md5: None
DEBUG:GEOparse:md5: None
56.4MB [00:02, 23.8MB/s]
21-Apr-2026 14:32:04 DEBUG downloader - Moving /tmp/tmp2uy0mpzh to /content/GPL13158.txt
DEBUG:GEOparse:Moving /tmp/tmp2uy0mpzh to /content/GPL13158.txt
21-Apr-2026 14:32:05 DEBUG downloader - Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL13158&form=text&view=full
DEBUG:GEOparse:Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL13158&form=text&view=full
21-Apr-2026 14:32:05 INFO GEOparse - Parsing ./GPL13158.txt: 
INFO:GEOparse:Parsing ./GPL13158.txt: 
21-Apr-2026 14:32:05 DEBUG GEOparse - PLATFORM: GPL13158
DEBUG:GEOparse:PLATFORM: GPL13158
/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringI


>>> Processing Series: GSE72747


100%|██████████| 23.4M/23.4M [00:03<00:00, 6.26MB/s]
21-Apr-2026 14:32:20 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
21-Apr-2026 14:32:20 DEBUG downloader - Moving /tmp/tmpbgv36njq to /content/GSE72747_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpbgv36njq to /content/GSE72747_family.soft.gz
21-Apr-2026 14:32:20 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE72nnn/GSE72747/soft/GSE72747_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE72nnn/GSE72747/soft/GSE72747_family.soft.gz
21-Apr-2026 14:32:20 INFO GEOparse - Parsing ./GSE72747_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE72747_family.soft.gz: 
21-Apr-2026 14:32:20 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
21-Apr-2026 14:32:20 DEBUG GEOparse - SERIES: GSE72747
DEBUG:GEOparse:SERIES: GSE72747
21-Apr-2026 14:32:20 DEBUG GEOparse - PLATFORM: GPL570
DEBUG:GEOparse:PLATFORM: GPL570
/usr/local


DONE! Dataset saved to database_robust.csv
